# Transfer API Current-Cycle Test

This notebook tests whether `https://api.cbbstat.com/players/transfers` returns current 2026 men's basketball transfer records and whether any players look available/uncommitted.

It uses the same logic as `scripts/update_transfer_portal.py`.

In [ ]:
from datetime import datetime, timezone
import json
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

API_URL = "https://api.cbbstat.com/players/transfers"
YEAR = 2026
url = f"{API_URL}?{urlencode({'year': YEAR})}"
url

## Fetch The API

If this cell errors with DNS/network failure, the environment cannot reach the API. If it succeeds, the following cells summarize whether 2026 records and available transfers exist.

In [ ]:
req = Request(url, headers={"User-Agent": "ncaa-player-dashboard/transfer-api-test"})
with urlopen(req, timeout=45) as response:
    payload = json.loads(response.read().decode("utf-8"))

raw = pd.DataFrame(payload)
raw.shape

## Normalize Availability

The API returns a destination field named `to`. The dashboard treats a player as available when `to` is blank or says something like uncommitted/undecided.

In [ ]:
available_values = {"", "na", "nan", "none", "uncommitted", "undecided", "tbd"}
df = raw.copy()
for col in ["id", "player", "from", "to", "exp", "year"]:
    if col not in df.columns:
        df[col] = ""

to_text = df["to"].fillna("").astype(str).str.strip().str.lower()
df["available"] = to_text.isin(available_values)
df["status"] = df["available"].map({True: "Available transfer", False: "Portal committed"})
df["tested_at_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")

summary = pd.Series({
    "api_url": url,
    "rows_returned": len(df),
    "year_min": pd.to_numeric(df["year"], errors="coerce").min(),
    "year_max": pd.to_numeric(df["year"], errors="coerce").max(),
    "available_count": int(df["available"].sum()),
    "committed_count": int((~df["available"]).sum()),
})
summary

## Available Transfers Returned By The API

In [ ]:
available = (
    df.loc[df["available"], ["id", "player", "from", "to", "exp", "year", "status", "tested_at_utc"]]
    .sort_values(["year", "player"], ascending=[False, True])
    .reset_index(drop=True)
)
available

## Committed / Historical Transfers Returned By The API

In [ ]:
committed = (
    df.loc[~df["available"], ["id", "player", "from", "to", "exp", "year", "status", "tested_at_utc"]]
    .sort_values(["year", "player"], ascending=[False, True])
    .reset_index(drop=True)
)
committed.head(50)

## Save Test Output

If the API succeeds, this writes the normalized 2026 response to `transfer_portal_2026_api_test.csv` for manual inspection.

In [ ]:
out_cols = ["id", "player", "from", "to", "exp", "year", "available", "status", "tested_at_utc"]
df[out_cols].to_csv("transfer_portal_2026_api_test.csv", index=False)
f"wrote {len(df)} rows"

## Result From This Codex Environment

I attempted the 2026 fetch from this workspace on 2026-05-23. The environment could not resolve `api.cbbstat.com`, even after requesting network escalation, so I could not verify the live API contents here.

Captured error:

```text
urllib.error.URLError: <urlopen error [Errno 8] nodename nor servname provided, or not known>
```

Run this notebook in a normal local Python/Jupyter environment to get the real answer.